In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y libassimp-dev
!pip install --upgrade "hyperdrone[examples]"

In [ ]:
import time

from hyperdrone import render
from hyperdrone.examples.data import procthor_scene_path

NUM_CAMERAS = 4096
WIDTH, HEIGHT = 64, 64
WARMUP_ITERATIONS = 10
ITERATIONS = 100

scene = render.load_scene(procthor_scene_path(), fidelity="medium")
renderer = render.Renderer(
    width=WIDTH, height=HEIGHT, num_cameras=NUM_CAMERAS,
    output="rgb", fidelity="medium",
)
renderer.init(scene)
renderer.generate_cameras(center=(-3.92, -5.67, 1.0), radius=0.01, fov=1.3962634015954636)

for _ in range(WARMUP_ITERATIONS):
    renderer.render_launch("rgb")
renderer.synchronize()

start = time.perf_counter()
for _ in range(ITERATIONS):
    renderer.render_launch("rgb")
renderer.synchronize()
elapsed = time.perf_counter() - start

frames = ITERATIONS * NUM_CAMERAS
print(f"Backend: {renderer.backend}")
print(f"Time: {elapsed:.3f} s ({elapsed * 1000 / ITERATIONS:.2f} ms/iteration)")
print(f"Throughput: {frames / elapsed:.1f} frames/s")
print(f"Rays: {frames * WIDTH * HEIGHT / 1e6 / elapsed:.1f} MRays/s")

In [ ]:
import matplotlib.pyplot as plt
import torch

frame = torch.from_dlpack(renderer.frame_dlpack())
print(f"Framebuffer: {frame.shape}, {frame.dtype}, {frame.device}")

GRID = 12
tiled = (frame[:GRID**2]
         .reshape(GRID, GRID, HEIGHT, WIDTH, 4)
         .permute(0, 2, 1, 3, 4)
         .reshape(GRID * HEIGHT, GRID * WIDTH, 4))
plt.figure(figsize=(12, 12))
plt.imshow(tiled.cpu())
plt.axis("off")
plt.show()